In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

# 생성형 AI 평가
- 첫번째 체인 : 국가명 → 해당 국가에서 가장 유명한 음식
- 두번째 체인 : 음식 → 해당 음식의 레시피
- 최종 체인 : 국가명 → 해당 국가에서 가장 유명한 음식의 레시피

### 파일명 : 생성형AI.ipynb, 생성형AI.html, 실행화면.png를 압축하여 "본인이름_생성형AI.zip으로 메일 첨부해주세요.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

In [ ]:
llm = ChatOllama(
    model='exaone3.5:2.4b',
    temperature=0.2,
    top_k=40,
    top_p=0.9,
)

outputParser = StrOutputParser()

In [ ]:
first_prompt = PromptTemplate(
    template='{나라}에서 가장 유명한 음식은 무엇인가요? 음식의 이름만 알려주세요.',
    input_variables = ['나라']
)

In [ ]:
first_chain = first_prompt | llm | outputParser

In [ ]:
second_prompt = PromptTemplate(
    template='{음식}의 레시피는 무엇인가요?',
    input_variables = ['음식']
)

In [ ]:
second_chain = second_prompt | llm | outputParser

In [ ]:
final_chain = first_chain | second_chain

In [ ]:
result = final_chain.invoke({'나라':'미국'})

In [ ]:
print(result)

---

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
first_prompt = PromptTemplate(
    template='{나라}에서 가장 유명한 음식은 무엇인가요? 음식의 이름만 알려주세요.',
    input_variables = ['나라']
)

class foodDetail(BaseModel):
    food:str = Field(description='해당 국가의 가장 유명한 음식')
        
structedllm = llm.with_structured_output(foodDetail)

food_chain = first_prompt|structedllm

food_chain.invoke({'나라': '이탈리아'})

In [ ]:
second_prompt = PromptTemplate(
    template='{음식}의 레시피는 무엇인가요? 섭씨와 ml, g의 단위를 사용해주세요',
    input_variables = ['음식']
)

class recipeDetail(BaseModel):
    recipe:str = Field(description='해당 음식의 가장 유명한 레시피')
        
structedllm = llm.with_structured_output(recipeDetail)

recipe_chain = second_prompt|structedllm

recipe_chain.invoke({'음식': '김치'})

In [ ]:
final_chain = food_chain | recipe_chain

country = input('국가명 입력 : ')

result = final_chain.invoke({'나라': country})

print(result.recipe)